In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import csv
import torch
from torch import nn
from torchsummary import summary
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, SubsetRandomSampler
from sklearn.model_selection import train_test_split
from torchvision.datasets import DatasetFolder
from torch.utils.data import Dataset
from PIL import Image

In [ ]:
class PairedImageDataset(Dataset):
    def __init__(self, real_images_path, target_images_path, transform=None):
        self.real_images_path = real_images_path
        self.target_images_path = target_images_path
        self.transform = transform
        
        # List all files in the directories
        self.real_images = sorted(os.listdir(real_images_path))
        self.target_images = sorted(os.listdir(target_images_path))

    def __len__(self):
        return len(self.real_images)

    def __getitem__(self, idx):
        # Load the real and target images
        real_image_path = os.path.join(self.real_images_path, self.real_images[idx])
        target_image_path = os.path.join(self.target_images_path, self.target_images[idx])
        
        real_image = Image.open(real_image_path).convert('RGB')
        target_image = Image.open(target_image_path).convert('RGB')

        # Apply transformations
        if self.transform:
            real_image = self.transform(real_image)
            target_image = self.transform(target_image)

        return real_image, target_image


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
real_images_path = "/kaggle/input/comic-faces-paired-synthetic-v2/face2comics_v2.0.0_by_Sxela/face2comics_v2.0.0_by_Sxela/faces"
target_images_path = "/kaggle/input/comic-faces-paired-synthetic-v2/face2comics_v2.0.0_by_Sxela/face2comics_v2.0.0_by_Sxela/comics"

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = PairedImageDataset(real_images_path, target_images_path, transform)

In [ ]:
train_split_size = 0.8
np.random.seed(36)

In [ ]:
dataset_size = len(dataset)
indices = np.arange(dataset_size)
np.random.shuffle(indices)

split = int(train_split_size * dataset_size)  # 80-20 train-test split
train_indices, test_indices = indices[:split], indices[split:]

train_sampler = SubsetRandomSampler(train_indices)
test_sampler = SubsetRandomSampler(test_indices)

train_loader = DataLoader(dataset, batch_size=48, sampler=train_sampler)
test_loader = DataLoader(dataset, batch_size=48, sampler=test_sampler)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels)
        )

    def forward(self, x):
        return x + self.block(x)  # Skip connection

In [ ]:
def downsample_block(in_channels, out_channels, kernel_size=4, stride=2, padding=1, use_batchnorm=True):
    layers = [
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=not use_batchnorm)
    ]
    if use_batchnorm:
        layers.append(nn.BatchNorm2d(out_channels))
    layers.append(nn.LeakyReLU(0.2, inplace=True))
    return nn.Sequential(*layers)

def upsample_block(in_channels, out_channels, kernel_size=4, stride=2, padding=1, use_dropout=False):
    layers = [
        nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride, padding, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    ]
    if use_dropout:
        layers.append(nn.Dropout(0.5))
    return nn.Sequential(*layers)

In [ ]:
class Generator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, num_residual_blocks=4):
        super(Generator, self).__init__()
        self.down_convolution_1 = downsample_block(in_channels, 64, use_batchnorm=False)
        self.down_convolution_2 = downsample_block(64, 128)
        self.down_convolution_3 = downsample_block(128, 256)
        self.down_convolution_4 = downsample_block(256, 512)
        self.down_convolution_5 = downsample_block(512, 512)
        self.down_convolution_6 = downsample_block(512, 512)
        self.down_convolution_7 = downsample_block(512, 512)
        self.down_convolution_8 = downsample_block(512, 512)

        self.res_blocks = nn.Sequential(*[ResidualBlock(512) for _ in range(num_residual_blocks)])
        
        self.up_convolution_1 = upsample_block(512, 512, use_dropout=True)
        self.up_convolution_2 = upsample_block(1024, 512, use_dropout=True)
        self.up_convolution_3 = upsample_block(1024, 512, use_dropout=True)
        self.up_convolution_4 = upsample_block(1024, 512)
        self.up_convolution_5 = upsample_block(1024, 256)
        self.up_convolution_6 = upsample_block(512, 128)
        self.up_convolution_7 = upsample_block(256, 64)

        self.final = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(128, out_channels, kernel_size=4, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        down_1 = self.down_convolution_1(x)
        down_2 = self.down_convolution_2(down_1)
        down_3 = self.down_convolution_3(down_2)
        down_4 = self.down_convolution_4(down_3)
        down_5 = self.down_convolution_5(down_4)
        down_6 = self.down_convolution_6(down_5)
        down_7 = self.down_convolution_7(down_6)
        down_8 = self.down_convolution_8(down_7)

        res = self.res_blocks(down_8)  # Pass through residual blocks

        up_1 = self.up_convolution_1(res)
        up_1 = torch.cat([up_1, down_7], 1)
        up_2 = self.up_convolution_2(up_1)
        up_2 = torch.cat([up_2, down_6], 1)
        up_3 = self.up_convolution_3(up_2)
        up_3 = torch.cat([up_3, down_5], 1)
        up_4 = self.up_convolution_4(up_3)
        up_4 = torch.cat([up_4, down_4], 1)
        up_5 = self.up_convolution_5(up_4)
        up_5 = torch.cat([up_5, down_3], 1)
        up_6 = self.up_convolution_6(up_5)
        up_6 = torch.cat([up_6, down_2], 1)
        up_7 = self.up_convolution_7(up_6)
        up_7 = torch.cat([up_7, down_1], 1)
        up_8 = self.final(up_7)
        return up_8

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0),
            nn.Sigmoid(),  # Add Sigmoid for BCELoss
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
criterion_GAN = nn.BCELoss()
criterion_pixelwise = nn.L1Loss()
# CHANGE THIS
target_ratio = 0.10

def compute_dynamic_lambda(L_GAN, L_pixel, target_ratio):
    epsilon = 1e-8  # Small constant for numerical stability
    lam = (target_ratio * L_GAN) / ((1 - target_ratio) * L_pixel + epsilon)
    return torch.clamp(lam, min=1e-3, max=100)  # Clamps lambda between 1e-3 and 100)

# Discriminator loss
def discriminator_loss(D_real, D_fake):
    real_loss = criterion_GAN(D_real, torch.ones_like(D_real))
    fake_loss = criterion_GAN(D_fake, torch.zeros_like(D_fake))
    return (real_loss + fake_loss) * 0.5

# Generator loss
def generator_loss(D_fake, gen_output, target):
    gan_loss = criterion_GAN(D_fake, torch.ones_like(D_fake))  # Fool the discriminator
    pixelwise_loss = criterion_pixelwise(gen_output, target)

    # Compute dynamic lambda
    lam = compute_dynamic_lambda(gan_loss, pixelwise_loss, target_ratio)

    return [gan_loss + (lam * pixelwise_loss), gan_loss, pixelwise_loss, lam]

In [ ]:
lr = 0.0002
beta1 = 0.5
beta2 = 0.999

# Initialize model and wrap it in DataParallel
generator = Generator().to(device)
discriminator = Discriminator().to(device)

# Load saved weights
# generator.load_state_dict(torch.load("/kaggle/input/generator29e/pytorch/default/1/generator_epoch29.pth", map_location=device))
# discriminator.load_state_dict(torch.load("/kaggle/input/discriminator29e/discriminator_epoch29.pth", map_location=device))

generator = nn.DataParallel(generator)
discriminator = nn.DataParallel(discriminator)

# Set models to training mode
generator.train()
discriminator.train()

optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, beta2))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, beta2))

In [ ]:
def print_and_save_images(tensor, num_images, epoch, iteration, title=None, save_dir="generated_images", filename_prefix="image"):
    os.makedirs(save_dir, exist_ok=True)

    tensor = tensor.detach().cpu().numpy()
    tensor = tensor.transpose((0, 2, 3, 1))  # Convert (N, C, H, W) → (N, H, W, C)
    
    fig, axes = plt.subplots(1, num_images, figsize=(num_images * 3, 3))  # Reduce figure height
    plt.subplots_adjust(top=0.8, bottom=0, left=0, right=1, wspace=0.1)  # Adjust layout

    full_title = f"{title} (Epoch {epoch}, Iteration {iteration})" if title else f"Epoch {epoch}, Iteration {iteration}"
    fig.suptitle(full_title, fontsize=12, y=0.9)  # Move title higher

    for i in range(num_images):
        image = tensor[i]
        if image.dtype.kind == 'f':  
            image = (image - image.min()) / (image.max() - image.min())  # Normalize
        
        axes[i].imshow(image, cmap='gray' if image.shape[-1] == 1 else None)
        axes[i].axis('off')

        save_path = os.path.join(save_dir, f"{filename_prefix}_epoch{epoch}_iter{iteration}_{i+1}.png")
        plt.imsave(save_path, image)

    plt.show()

In [ ]:
def train(num_epochs=10):

    loss_file = '/kaggle/working/losses.csv'
    
    # Create CSV file and write headers if it doesn't exist
    if not os.path.exists(loss_file):
        with open(loss_file, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["Epoch", "Step", "D Loss", "G Loss", "Gan Loss", "Pixelwise Loss", "Lambda"])
            
    for epoch in range(num_epochs):
        epoch_progress = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{num_epochs}", position=0, leave=True)

        total_batches = len(train_loader)
        selected_iters = [0, 1, 2]
        
        for i, (real_images, target_images) in epoch_progress:
            real_images = real_images.to(device)
            target_images = target_images.to(device)

            # Generate fake images
            fake_images = generator(real_images)

            # Train Discriminator
            optimizer_D.zero_grad()
            D_real = discriminator(torch.cat([real_images, target_images], 1))
            D_fake = discriminator(torch.cat([real_images, fake_images.detach()], 1))
            D_loss = discriminator_loss(D_real, D_fake)
            D_loss.backward()
            optimizer_D.step()

            # Train Generator
            optimizer_G.zero_grad()
            D_fake = discriminator(torch.cat([real_images, fake_images], 1))
            G_loss = generator_loss(D_fake, fake_images, target_images)
            G_loss[0].backward()
            optimizer_G.step()

            # Save losses to CSV file
            with open(loss_file, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([epoch+1, i, D_loss.item(), G_loss[0].item(), G_loss[1].item(), G_loss[2].item(), G_loss[3].item()])

            # Update tqdm progress bar with loss values
            epoch_progress.set_postfix({"D Loss": D_loss.item(), "G Loss": G_loss[0].item(), "GanLoss:": G_loss[1].item(), "PixelLoss": G_loss[2].item(), "Lambda": G_loss[3].item()})
            if epoch == 50 and i in selected_iters:
                print_and_save_images(real_images, 5, epoch, i, title="Real Images", save_dir="real_images", filename_prefix="real")
                print_and_save_images(fake_images, 5, epoch, i, title="Generated Images", save_dir="generated_images", filename_prefix="fake")
                print_and_save_images(target_images, 5, epoch, i, title="Target Images", save_dir="target_images", filename_prefix="target")

        print_and_save_images(real_images, 5, epoch, i, title="Real Images", save_dir="real_images", filename_prefix="real")
        print_and_save_images(fake_images, 5, epoch, i, title="Generated Images", save_dir="generated_images", filename_prefix="fake")
        print_and_save_images(target_images, 5, epoch, i, title="Target Images", save_dir="target_images", filename_prefix="target")

        os.makedirs("/kaggle/working/models/generator", exist_ok=True)
        os.makedirs("/kaggle/working/models/discriminator", exist_ok=True)
                
        # Save model
        if epoch > 29:
            torch.save(generator.module.state_dict() if hasattr(generator, "module") else generator.state_dict(), 
                               f'/kaggle/working/models/generator/generator_epoch{epoch}.pth')
            torch.save(discriminator.module.state_dict() if hasattr(discriminator, "module") else discriminator.state_dict(), 
                               f'/kaggle/working/models/discriminator/discriminator_epoch{epoch}.pth')
            torch.save(optimizer_G.state_dict(), f'optimizer_G{epoch}.pth')
            torch.save(optimizer_D.state_dict(), f'optimizer_D{epoch}.pth')
            


In [ ]:
train(num_epochs=51)

In [ ]:
import os
import shutil
from IPython.display import FileLink

# Define working directory
working_dir = "/kaggle/working/"

# List all folders in the working directory
folders = [f for f in os.listdir(working_dir) if os.path.isdir(os.path.join(working_dir, f))]

# Zip and generate download links for each folder
for folder in folders:
    folder_path = os.path.join(working_dir, folder)
    zip_path = os.path.join(working_dir, f"{folder}.zip")
    
    # Zip the folder
    shutil.make_archive(zip_path.replace(".zip", ""), 'zip', folder_path)
    
    # Provide download link
    display(FileLink(zip_path))

FileLink('/kaggle/working/losses.csv')

In [ ]:
"""
import shutil
folders = os.listdir('/kaggle/working/')
for folder in folders:
    shutil.rmtree(f'/kaggle/working/{folder}')
"""